In [20]:
import numpy as np 
import pandas as pd 
import seaborn as sns 
import matplotlib.pyplot as plt
import sklearn
import time 

In [3]:
df  = pd.read_csv("Crop Recommendation dataset(npk dataset 1).csv")

In [4]:
df.head()

,N,P,K,temperature,humidity,ph,rainfall,label
0,90,42,43,20.879744,82.002744,6.502985,202.935536,rice
1,85,58,41,21.770462,80.319644,7.038096,226.655537,rice
2,60,55,44,23.004459,82.320763,7.840207,263.964248,rice
3,74,35,40,26.491096,80.158363,6.980401,242.864034,rice
4,78,42,42,20.130175,81.604873,7.628473,262.717340,rice


In [5]:
X = df.drop('label', axis = 1)
y = df.label

In [6]:
X

,N,P,K,temperature,humidity,ph,rainfall
0,90,42,43,20.879744,82.002744,6.502985,202.935536
1,85,58,41,21.770462,80.319644,7.038096,226.655537
2,60,55,44,23.004459,82.320763,7.840207,263.964248
3,74,35,40,26.491096,80.158363,6.980401,242.864034
4,78,42,42,20.130175,81.604873,7.628473,262.717340
...,...,...,...,...,...,...,...
2195,107,34,32,26.774637,66.413269,6.780064,177.774507
2196,99,15,27,27.417112,56.636362,6.086922,127.924610
2197,118,33,30,24.131797,67.225123,6.362608,173.322839
2198,117,32,34,26.272418,52.127394,6.758793,127.175293


In [7]:
y

0         rice
1         rice
2         rice
3         rice
4         rice
         ...  
2195    coffee
2196    coffee
2197    coffee
2198    coffee
2199    coffee
Name: label, Length: 2200, dtype: str

In [11]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score

# 1. Standardize Feature Names and Scale
X.columns = X.columns.str.strip().str.lower()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Encode Labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 3. The 70/15/15 Split
X_train, X_temp, y_train, y_temp = train_test_split(X_scaled, y_encoded, test_size=0.3, random_state=42)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42)

# 4. Define the Tournament Function
def evaluate_model(model, name):
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    
    print(f"--- {name} Results ---")
    print(f"Accuracy: {accuracy_score(y_val, preds):.4f}")
    print(f"Precision: {precision_score(y_val, preds, average='macro'):.4f}")
    print(f"Recall: {recall_score(y_val, preds, average='macro'):.4f}")
    print(f"F1-Score: {f1_score(y_val, preds, average='macro'):.4f}")
    print(f"Kappa: {cohen_kappa_score(y_val, preds):.4f}\n")

# 5. The Full Tournament Expansion
print("--- STARTING THE EXPANDED TOURNAMENT ---")

# The Linear Baseline
evaluate_model(LogisticRegression(max_iter=1000), "Logistic Regression")

# The Heavyweight Champion
evaluate_model(RandomForestClassifier(n_estimators=100), "Random Forest")

# The Geometric Fighter (SVM)
# We test 'linear' for simple boundaries and 'rbf' for complex curves
evaluate_model(SVC(kernel='linear'), "SVM (Linear)")
evaluate_model(SVC(kernel='rbf'), "SVM (RBF)")

# The Probabilistic Fighter (Naive Bayes)
evaluate_model(GaussianNB(), "Naive Bayes")

--- STARTING THE EXPANDED TOURNAMENT ---
--- Logistic Regression Results ---
Accuracy: 0.9667
Precision: 0.9703
Recall: 0.9686
F1-Score: 0.9678
Kappa: 0.9650

--- Random Forest Results ---
Accuracy: 0.9909
Precision: 0.9932
Recall: 0.9909
F1-Score: 0.9913
Kappa: 0.9905

--- SVM (Linear) Results ---
Accuracy: 0.9727
Precision: 0.9768
Recall: 0.9733
F1-Score: 0.9727
Kappa: 0.9714

--- SVM (RBF) Results ---
Accuracy: 0.9697
Precision: 0.9743
Recall: 0.9729
F1-Score: 0.9707
Kappa: 0.9682

--- Naive Bayes Results ---
Accuracy: 0.9939
Precision: 0.9952
Recall: 0.9939
F1-Score: 0.9942
Kappa: 0.9936



In [13]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, cohen_kappa_score

# 1. Clean Column Names and Scale Features
X.columns = X.columns.str.strip().str.lower()
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# 2. Encode Labels
le = LabelEncoder()
y_encoded = le.fit_transform(y)

# 3. Standard Split (70/15/15) for single-point metrics
X_train, X_remainder, y_train, y_remainder = train_test_split(
    X_scaled, y_encoded, test_size=0.3, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_remainder, y_remainder, test_size=0.5, random_state=42
)

# 4. Define the Tournament Function with K-Fold
results_list = []

def evaluate_model(model, name):
    # Perform 5-Fold Cross Validation on the full dataset
    # This gives us the most honest look at "Overfitting"
    cv_scores = cross_val_score(model, X_scaled, y_encoded, cv=5)
    
    # Train and test once on the validation set for the other 4 metrics
    model.fit(X_train, y_train)
    preds = model.predict(X_val)
    
    # Calculate Metrics
    acc = accuracy_score(y_val, preds)
    prec = precision_score(y_val, preds, average='macro')
    rec = recall_score(y_val, preds, average='macro')
    f1 = f1_score(y_val, preds, average='macro')
    kappa = cohen_kappa_score(y_val, preds)
    
    # Store results
    results_list.append({
        "Model": name,
        "CV Mean Acc (%)": round(cv_scores.mean() * 100, 2),
        "CV Std Dev": round(cv_scores.std(), 4),
        "Val Accuracy (%)": round(acc * 100, 2),
        "Precision": round(prec, 4),
        "Recall": round(rec, 4),
        "F1-Score": round(f1, 4),
        "Kappa": round(kappa, 4)
    })

# 5. Run the Final Tournament
print("--- STARTING K-FOLD CROSS-VALIDATION TOURNAMENT ---")
evaluate_model(LogisticRegression(max_iter=1000), "Logistic Regression")
evaluate_model(RandomForestClassifier(n_estimators=100, random_state=42), "Random Forest")
evaluate_model(SVC(kernel='linear'), "SVM (Linear)")
evaluate_model(SVC(kernel='rbf'), "SVM (RBF)")
evaluate_model(GaussianNB(), "Naive Bayes")

# 6. Display the Final Leaderboard
df_results = pd.DataFrame(results_list)
# We sort by CV Mean Accuracy because it's the most reliable metric
df_results = df_results.sort_values(by="CV Mean Acc (%)", ascending=False)

display(df_results)

--- STARTING K-FOLD CROSS-VALIDATION TOURNAMENT ---


,Model,CV Mean Acc (%),CV Std Dev,Val Accuracy (%),Precision,Recall,F1-Score,Kappa
1,Random Forest,99.50,0.0036,98.79,0.9913,0.9879,0.9882,0.9873
4,Naive Bayes,99.50,0.0022,99.39,0.9952,0.9939,0.9942,0.9936
2,SVM (Linear),98.23,0.0049,97.27,0.9768,0.9733,0.9727,0.9714
3,SVM (RBF),98.23,0.0017,96.97,0.9743,0.9729,0.9707,0.9682
0,Logistic Regression,97.14,0.0099,96.67,0.9703,0.9686,0.9678,0.9650


In [24]:
import time
import numpy as np
import warnings

# Mute the "Feature Names" warning clutter
warnings.filterwarnings("ignore", category=UserWarning)

def run_agri_advisor_v2():
    print("\n" + "="*50)
    print("🌾  360° AGRI: INTERACTIVE CROP ADVISOR  🌾")
    print("="*50)
    
    try:
        # 1. Capture Inputs from the VS Code Top Bar
        n = float(input("➡️  Enter Nitrogen (N): "))
        p = float(input("➡️  Enter Phosphorus (P): "))
        k = float(input("➡️  Enter Potassium (K): "))
        temp = float(input("➡️  Enter Temperature (°C): "))
        hum = float(input("➡️  Enter Humidity (%): "))
        ph = float(input("➡️  Enter Soil pH: "))
        rain = float(input("➡️  Enter Rainfall (mm): "))

        # 2. LOG THE INPUTS (This makes them visible in your output cell)
        print("\n" + "-"*15 + " INPUT SUMMARY " + "-"*15)
        print(f"📍 Nitrogen (N):    {n}")
        print(f"📍 Phosphorus (P):  {p}")
        print(f"📍 Potassium (K):   {k}")
        print(f"📍 Temperature:     {temp}°C")
        print(f"📍 Humidity:        {hum}%")
        print(f"📍 Soil pH:         {ph}")
        print(f"📍 Rainfall:        {rain}mm")
        print("-" * 45)

        print("\n🔍 Analyzing soil and weather patterns...")
        time.sleep(1.0)

        # 3. Model Logic
        input_data = np.array([[n, p, k, temp, hum, ph, rain]])
        scaled_input = scaler.transform(input_data)
        prediction_idx = final_model.predict(scaled_input)[0]
        probs = final_model.predict_proba(scaled_input)[0]

        # 4. Results Display
        recommended_crop = le.inverse_transform([prediction_idx])[0]
        confidence = round(np.max(probs) * 100, 2)

        print(f"\n✨ RESULT: The most suitable crop is **{recommended_crop.upper()}**")
        print(f"📈 MODEL CONFIDENCE: {confidence}%")
        print("="*50)

    except ValueError:
        print("\n❌ INPUT ERROR: Please enter numerical values only.")

# Execute
run_agri_advisor_v2()


🌾  360° AGRI: INTERACTIVE CROP ADVISOR  🌾

--------------- INPUT SUMMARY ---------------
📍 Nitrogen (N):    40.0
📍 Phosphorus (P):  60.0
📍 Potassium (K):   80.0
📍 Temperature:     18.5°C
📍 Humidity:        16.0%
📍 Soil pH:         7.5
📍 Rainfall:        85.0mm
---------------------------------------------

🔍 Analyzing soil and weather patterns...

✨ RESULT: The most suitable crop is **CHICKPEA**
📈 MODEL CONFIDENCE: 100.0%
